# 07 — v4: v3 with a memory-safe two-process run

Same model as `06_v3_france` (GPU bi-encoder + cross-encoder + LightGBM + France self-training). The training stage
saves every model to `artifacts/` and the test stage runs in a fresh Python process that loads them, so no training
memory carries into the test phase. Embeddings are written into a preallocated GPU tensor chunk by chunk.


In [ ]:
import sys, json, resource, platform
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from config import ARTIFACT_DIR, DATA_DIR, OUTPUT_DIR, ON_KAGGLE
from pipeline import RunConfig, run
pd.set_option("display.width", 200)
print(f"data {DATA_DIR}\noutput {OUTPUT_DIR}\nartifacts {ARTIFACT_DIR}\non Kaggle: {ON_KAGGLE}")
cfg = RunConfig(use_neural=True, france_self_train=True)
cfg

In [ ]:
report = run(cfg, OUTPUT_DIR, ARTIFACT_DIR)
print('neural stages used:', report['neural'])
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"peak memory {peak / (1e9 if platform.system() == 'Darwin' else 1e6):.1f} GB")

## Validation (official macro F0.5 on the validation fold)

In [ ]:
pd.Series(report["validation"]).to_frame("value")

In [ ]:
pd.DataFrame(report["blocking"]).T

In [ ]:
table = pd.read_csv(ARTIFACT_DIR / "threshold_table.tsv", sep="\t")
ax = table.plot(x="threshold", y=["f05", "precision", "recall"], figsize=(9, 4), marker=".")
ax.axvline(report["threshold"], color="grey", ls="--"); ax.set_title(f"validation F0.5 vs threshold (chosen {report['threshold']})")
plt.show()
table.round(4)

## Most important features (gain)

In [ ]:
pd.Series(report["top_features"]).to_frame("gain")

## Test output

In [ ]:
print(json.dumps(report["test"], indent=2))
for f in ("matching_results.tsv", "candidate_pairs.tsv"):
    p = OUTPUT_DIR / f
    print(f"{f}: {p.stat().st_size / 1e6:.1f} MB")
    display(pd.read_csv(p, sep="\t", nrows=5, keep_default_na=False))

In [ ]:
print(json.dumps(report['test'].get('france_self_training'), indent=2))
